# M06 Tracing the Thoughts of a Large Language Model


## Read a local attribution graph

Load a teaching artifact and inspect which paths contribute most strongly to the final answer.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/circuits-zoom-in-fresh-20260325.git"
REPO_DIR = "circuits-zoom-in-fresh-20260325"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import json
import matplotlib.pyplot as plt

graph = json.loads((root / "artifacts" / "m06_attribution_graph.json").read_text())
case = graph["cases"][0]

fig, ax = plt.subplots(figsize=(10, 4))
for edge in case["edges"]:
    source = next(node for node in case["nodes"] if node["id"] == edge["source"])
    target = next(node for node in case["nodes"] if node["id"] == edge["target"])
    ax.plot(
        [source["x"], target["x"]],
        [source["y"], target["y"]],
        linewidth=2 + 4 * edge["score"],
        color="#c96a28",
        alpha=0.65,
    )

for node in case["nodes"]:
    color = "#123b63" if node["kind"] == "feature" else "#b5893a"
    ax.scatter(node["x"], node["y"], s=700, color=color)
    ax.text(node["x"], node["y"], node["label_en"], ha="center", va="center", color="white", fontsize=9)

ax.set_title(case["title_en"])
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
plt.tight_layout()

sorted_edges = sorted(case["edges"], key=lambda edge: edge["score"], reverse=True)
print("Top contributions:")
for edge in sorted_edges:
    print(f"  {edge['source']} -> {edge['target']}: {edge['score']:.2f}")


## Takeaway

Tracing is about finding a faithful slice of computation, not about dumping the whole network.


## Turn this notebook into research output

Research worksheet means this notebook is not complete when the cells finish. Use the templates in /templates and leave behind written outputs.


### Before you run

- Choose one path or subgraph you plan to explain in detail.
- Write down what a faithful slice of computation means in this context.
- Open the memo template and reserve a section called 'what the graph does not prove'.


### After you run

- Explain three high-contribution edges in plain language.
- Mark one ambiguity that would require a follow-up experiment.


### Ship these artifacts

- One graph walkthrough.
- One ambiguity note.
- One next experiment that would reduce uncertainty about the graph.


## Self-check questions

- Without simply reading back the labels, explain what the three most important contribution paths in this attribution graph are doing.
- What conclusion does the graph support, and what conclusion does it clearly not support?
- If you could run only one follow-up experiment to reduce the ambiguity in this graph, what would you choose?
- If you cannot answer at least two of these without rereading the note, revisit the article and your written outputs.
